# NCRB scanned tables: cloud OCR, round 2

Round 1 read all 12,758 scanned pages with OvisOCR2 at 1600 px. This round re-reads the **2,399 scanned files (9,967 pages) whose reading is still weak**: no tables found, nothing to check against, or under 98% of NCRB's printed totals reproduced. Two independent readings are made:

1. **OvisOCR2 at 2048 px:** the round-1 winner, with a larger page so small typed digits are clearer.
2. **PaddleOCR-VL-1.6 at 2048 px:** a different model, so its mistakes are independent of OvisOCR2's.

Back on the laptop each file keeps the reading that reproduces the most printed totals and internal sums (row totals, subtotals); where nothing can be checked, the reading the others agree with most.

**Before you start:** Runtime → Change runtime type → **A100 GPU**. Expect about 6 hours (about 35 compute units); the job stops itself at 9 hours.

**Run the three cells in order.** Cell 1 asks once for access to your Google Drive. Cell 2 starts the job in the background, detached from the notebook. Cell 3 shows progress, downloads `round2.zip` when finished and releases the GPU.

**Safe to do:** stop or re-run cell 3 at any time. **Don't:** restart the session or re-run cell 2 while the job runs (it checks and won't start twice, but a restart wipes the VM). If the session drops anyway, run cells 1–3 again: every page already read is on Drive and is skipped.

## 1. Drive, code and vLLM (3–5 minutes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/NCRB && git -C /content/NCRB pull -q || git clone -q --depth 1 https://github.com/reclaimchennai/NCRB.git /content/NCRB
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q -U vllm pymupdf requests beautifulsoup4 lxml tqdm 2>&1 | tail -2
import subprocess
cap = float(subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], capture_output=True, text=True).stdout.split()[0])
DTYPE = 'auto' if cap >= 8 else 'half'   # a T4 has no bfloat16
print('ready; dtype', DTYPE)

## 2. Start round 2 in the background

Returns in about 20 seconds. The job keeps running on its own after this.

In [ ]:
import subprocess, time
def job_pids():
    return subprocess.run(['pgrep', '-f', '[c]olab/run_all.py'], capture_output=True, text=True).stdout.split()
if job_pids():
    print('job already running, pid', job_pids())
else:
    subprocess.Popen(['python', 'colab/run_all.py', '--plan', 'colab/plans/round2.json', '--state', 'round2-status.json',
                      '--hours', '9', '--dtype', DTYPE],
                     cwd='/content/NCRB', stdout=open('/content/drive/MyDrive/ncrb_round2.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)   # detached: no cell can stop it
    time.sleep(20)
    print('job started, pid', job_pids() or 'NOT FOUND: see /content/drive/MyDrive/ncrb_round2.log')

## 3. Progress

One bar per reading. The line shows pages done, speed, time left, pages that failed (`err`), the GPU, **`totals`** (the share of NCRB's printed totals the new tables reproduce, next to the current reading of the same files) and compute units used. At the end the zip downloads to your computer and the GPU is released.

In [ ]:
!cd /content/NCRB && git pull -q
exec(open('/content/NCRB/colab/monitor.py').read())
watch(state='round2-status.json')

## If something looks wrong

Run this (it only reads) and paste the output:

In [ ]:
import json, subprocess
s = json.load(open('/content/drive/MyDrive/ncrb_ocr/round2-status.json'))
print({k: s.get(k) for k in ('phase', 'stage', 'full', 'last_error', 'round2_done')})
print(subprocess.run("pgrep -af '[c]olab/run_all.py|[o]cr_pages.py' | cut -c1-160; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader; tail -5 /content/drive/MyDrive/ncrb_round2.log | cut -c1-200",
                     shell=True, capture_output=True, text=True).stdout)

## When it finishes

`round2.zip` is in your Downloads folder (and in Drive under `ncrb_ocr/`). Tell Claude, or on the laptop run:

```
uv run python -m ncrb.vlm_import ~/Downloads/round2.zip
uv run python -m ncrb.extract --vlm-rescore
```